# Gün 15 - Native RAG (2): İstem, Üretim ve Test Senaryoları

İlk notebook'ta hattın arama yarısını kurdum: belgeyi parçaladım, yerel bir embedding modeliyle
vektörleştirdim, kosinüs benzerliğiyle en alakalı parçaları getiren fonksiyonu yazdım
ve alakasız soruları eleyen bir skor eşiği belirledim.

Bu notebook'ta ikinci yarıyı kuruyorum:

```
getirilen parçalar + soru -> istem -> LLM -> cevap
```

Ve asıl önemlisi, sistemi test ediyorum. Görevlendirmenin istediği iki senaryo var:

1. Belge içi sorulara **doğru** cevap vermeli
2. Belge dışı sorularda **halüsinasyon görmemeli**, "Bilgi bulunamadı" demeli

**Üretim modeli olarak Ollama üzerinden yerel bir model kullanıyorum.** İlk notebook'ta
embedding için yerele geçme gerekçemi yazmıştım; üretim tarafında sebep daha da net:
bulut API'sinin ücretsiz katmanında günlük sınır 20 istekti, oysa aşağıdaki testler
bunun üç katı çağrı yapıyor.

Ollama modeli 4-bit sıkıştırılmış (quantized) halde tutuyor. 7.6 milyar parametreli
model böylece yaklaşık 4.7 GB'a iniyor ve 6 GB'lık ekran kartıma sığıyor. Sıkıştırma
biraz kalite kaybettiriyor ama modeli çalıştırılabilir kılan da bu.

> **Not:** Bu notebook'un çalışması için arka planda Ollama servisinin açık olması ve
> `qwen2.5:7b-instruct` modelinin indirilmiş olması gerekiyor.

In [1]:
import json
import time
import urllib.request
import warnings
import numpy as np
import torch

warnings.filterwarnings("ignore")
np.random.seed(42)

OLLAMA = "http://localhost:11434"
URETIM_MODEL = "qwen2.5:7b-instruct"

# servis açık mı ve model var mı, kontrol ediyorum
with urllib.request.urlopen(f"{OLLAMA}/api/tags", timeout=10) as f:
    modeller = json.loads(f.read())["models"]

print("Ollama çalışıyor. Yüklü modeller:")
for m in modeller:
    ay = m["details"]
    print(f"  {m['name']:26} {ay['parameter_size']:>6}  {ay['quantization_level']:>8}"
          f"  {m['size']/1024**3:.2f} GB")

print()
print("cuda    :", torch.cuda.is_available())

Ollama çalışıyor. Yüklü modeller:
  qwen2.5:7b-instruct          7.6B    Q4_K_M  4.36 GB

cuda    : True


## 1. Adım - Kaydettiğim indeksi yüklüyorum

Parçalama ve vektörleştirmeyi baştan yapmıyorum, ilk notebook'ta kaydettiğim `indeks.npz`
dosyasını okuyorum. Gerçek sistemlerde de indeksleme bir kez yapılıp saklanıyor,
her sorguda tekrarlanmıyor.

In [2]:
d = np.load("indeks.npz", allow_pickle=True)

V_norm      = d["V_norm"]
parcalar    = list(d["parcalar"])
ESIK        = float(d["esik"])
EMBED_MODEL = str(d["yontem"])

print("indeks yüklendi")
print(f"  parça sayısı  : {len(parcalar)}")
print(f"  vektör boyutu : {V_norm.shape[1]}")
print(f"  eşik          : {ESIK:.4f}")
print(f"  embedding     : {EMBED_MODEL}")

indeks yüklendi
  parça sayısı  : 8
  vektör boyutu : 768
  eşik          : 0.8103
  embedding     : intfloat/multilingual-e5-base


Arama fonksiyonunu ilk notebook'taki haliyle kuruyorum. Sorunun vektörünü **aynı modelle** üretmem
şart, yoksa aynı uzayda olmazlar ve karşılaştırma anlamsız olur. e5 modeli sorulara
`query:` öneki istiyordu, onu da uyguluyorum.

In [3]:
from sentence_transformers import SentenceTransformer

CIHAZ = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer(EMBED_MODEL, device=CIHAZ)


def ara(soru, k=3):
    v = embed_model.encode([f"query: {soru}"], show_progress_bar=False)[0]
    v = v / np.linalg.norm(v)
    skorlar = V_norm @ v
    en_iyi = np.argsort(skorlar)[::-1][:k]
    return [(int(i), float(skorlar[i]), parcalar[i]) for i in en_iyi]


print("embedding modeli hazır, cihaz:", CIHAZ)
print("örnek arama:", f"skor = {ara('yıllık izin', k=1)[0][1]:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13285.79it/s]

embedding modeli hazır, cihaz: cuda


örnek arama: skor = 0.8682


## 2. Adım - İstemi kuruyorum

Hattın en kritik adımı burası. Gün 14'te halüsinasyon engelleme başlığında yazdığım
maddelerin hepsini bu isteme koyuyorum:

**1. Bağlama sadakat kuralı.** "Yalnızca aşağıdaki bağlamı kullan." Modelin eğitim
verisinden cevap üretmesini engelliyorum.

**2. Kaçış yolu.** "Bağlamda yoksa 'Bilgi bulunamadı' de." Gün 14'te öğrendiğim en
önemli madde buydu: modele bilmiyorum diyebileceği bir çıkış tanımlanmazsa, model bir
şey uydurmak zorunda hissediyor.

**3. Bağlamı sınırlarla ayırma.** Parçaları `<baglam>` etiketleri arasına alıyorum.
Böylece model talimat ile veriyi karıştırmıyor.

**4. Kaynak gösterme.** Her parçayı numaralandırıp modelden hangisini kullandığını
belirtmesini istiyorum.

Sohbet modelleri düz metin değil, **rol ayrımı olan mesaj listesi** bekliyor: sistem
mesajı kuralları, kullanıcı mesajı bağlam ve soruyu taşıyor. Bu ayrım işe yarıyor
çünkü model sistem mesajını "kim olduğum ve neye uyacağım", kullanıcı mesajını ise
"üzerinde çalışacağım veri" olarak öğrenmiş durumda.

In [4]:
SISTEM_TALIMATI = """Sen bir şirket içi asistanısın. Görevin, çalışan el kitabından
alınan bağlam parçalarını kullanarak soruları cevaplamak.

KURALLAR:
1. Cevabını YALNIZCA <baglam> etiketleri arasındaki metne dayandır.
2. Bağlamda cevap yoksa tam olarak şunu yaz: Bilgi bulunamadı
3. Kendi genel bilgini KULLANMA, tahmin YÜRÜTME.
4. Cevabın sonunda kullandığın parça numarasını [Parça N] biçiminde belirt.
5. Kısa ve net cevap ver, tek cümleyi geçme."""


def baglam_kur(sonuclar):
    bloklar = []
    for idx, skor, parca in sonuclar:
        temiz = " ".join(parca.split())
        bloklar.append(f"[Parça {idx}] (benzerlik: {skor:.3f})\n{temiz}")
    return "\n\n".join(bloklar)


def mesaj_kur(soru, sonuclar, talimat=SISTEM_TALIMATI):
    return [
        {"role": "system", "content": talimat},
        {"role": "user",
         "content": f"<baglam>\n{baglam_kur(sonuclar)}\n</baglam>\n\nSORU: {soru}"},
    ]


ornek = "Yıllık ücretli izin kaç gündür?"
mesajlar = mesaj_kur(ornek, ara(ornek, k=2))

print("--- SISTEM MESAJI ---")
print(mesajlar[0]["content"])
print()
print("--- KULLANICI MESAJI ---")
print(mesajlar[1]["content"][:650])

--- SISTEM MESAJI ---
Sen bir şirket içi asistanısın. Görevin, çalışan el kitabından
alınan bağlam parçalarını kullanarak soruları cevaplamak.

KURALLAR:
1. Cevabını YALNIZCA <baglam> etiketleri arasındaki metne dayandır.
2. Bağlamda cevap yoksa tam olarak şunu yaz: Bilgi bulunamadı
3. Kendi genel bilgini KULLANMA, tahmin YÜRÜTME.
4. Cevabın sonunda kullandığın parça numarasını [Parça N] biçiminde belirt.
5. Kısa ve net cevap ver, tek cümleyi geçme.

--- KULLANICI MESAJI ---
<baglam>
[Parça 2] (benzerlik: 0.886)
. Ekip toplantısı olan salı günleri ofiste bulunmak gereklidir. 2. İZİN HAKLARI Yıllık ücretli izin süresi, çalışanın kıdemine göre belirlenir. Bir ile beş yıl arası kıdemi olan çalışanlar 14 gün, beş ile on beş yıl arası kıdemi olanlar 20 gün, on beş yıldan fazla kıdemi olanlar 26 gün yıllık izin kullanır. Yıllık izin talepleri en az on gün önceden insan kaynakları sistemine girilmelidir. Mazeret izni yılda toplam 5 gündür ve önceden bildirim şartı aranmaz. Evlilik izni 3 gün,

## 3. Adım - Üretim fonksiyonu

Gün 14'ün sonunda parametre kararımı vermiştim: **sıcaklık sıfır, tekrar cezası yok.**
Gerekçesi şuydu: RAG'de amaç yaratıcılık değil bağlama sadakat.

Burada `temperature: 0.0` veriyorum. Gün 14'te gördüğüm gibi bu, dağılımı en olası
tokene kilitliyor; örnekleme yapılmıyor.

Bunu yazarken tekrarlanabilirlik konusunda yanıldığımı fark ettim. "Sıcaklık sıfırsa
aynı girdi hep aynı çıktıyı verir" diye düşünmüştüm; aynı soruyu iki ayrı oturumda
sorunca farklı cevaplar aldım. Sebebi örnekleme değil, GPU'da kayan nokta
işlemlerinin toplama sırasına duyarlı olması: aynı hesap farklı sırayla yapılınca
son basamaklarda minik farklar çıkıyor ve iki token birbirine çok yakınsa seçim
değişebiliyor. Yani sıcaklık sıfır **rastgeleliği** kaldırıyor ama **birebir aynı
çıktıyı** garanti etmiyor.

Ollama'ya HTTP üzerinden JSON gönderiyorum. Ek bir kütüphane kurmak yerine Python'un
kendi `urllib` modülünü kullanıyorum.

In [5]:
_onbellek = {}


def uret(mesajlar, maks_token=120):
    """Ollama'ya sohbet isteği gönderir. temperature=0 -> tekrarlanabilir."""
    anahtar = json.dumps(mesajlar, ensure_ascii=False)
    if anahtar in _onbellek:
        return _onbellek[anahtar]

    govde = json.dumps({
        "model": URETIM_MODEL,
        "messages": mesajlar,
        "stream": False,
        "options": {"temperature": 0.0, "num_predict": maks_token},
    }).encode("utf-8")

    istek = urllib.request.Request(f"{OLLAMA}/api/chat", data=govde,
                                   headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(istek, timeout=300) as f:
        cevap = json.loads(f.read())["message"]["content"].strip()

    _onbellek[anahtar] = cevap
    return cevap


t0 = time.time()
print("cevap:", uret(mesajlar))
print(f"süre : {time.time() - t0:.1f} sn")

cevap: Bilgi bulunamadı [Parça 2].
süre : 12.4 sn


## 4. Adım - Hattı birleştiriyorum

Bütün adımları tek fonksiyonda topluyorum:

```
soru -> ara -> skor eşiği kontrolü
                    |
        eşiğin altında mı? -> EVET -> "Bilgi bulunamadı" (model hiç çağrılmaz)
                    |
                  HAYIR
                    |
              istem kur -> model -> cevap
```

Kritik tasarım kararı: **eşik kontrolü modelden önce.** Alakasız soruda model hiç
çağrılmıyor. Gün 14'te bunu "iki savunma hattı" diye not etmiştim: birincisi skor
eşiği, ikincisi istemdeki kural.

In [6]:
def cevapla(soru, k=3, talimat=SISTEM_TALIMATI):
    sonuclar = ara(soru, k=k)
    en_yuksek = sonuclar[0][1]

    # 1. savunma hattı: skor eşiği
    if en_yuksek < ESIK:
        return {"soru": soru, "cevap": "Bilgi bulunamadı", "skor": en_yuksek,
                "kaynak": None, "model_cagrildi": False}

    secilen = [(i, s, p) for i, s, p in sonuclar if s >= ESIK * 0.97]

    # 2. savunma hattı: istemdeki sadakat kuralı
    cevap = uret(mesaj_kur(soru, secilen, talimat))

    return {"soru": soru, "cevap": cevap, "skor": en_yuksek,
            "kaynak": [i for i, _, _ in secilen], "model_cagrildi": True}


r = cevapla("Yemek kartına günlük ne kadar yükleniyor?")
for a, v in r.items():
    print(f"{a:15}: {v}")

soru           : Yemek kartına günlük ne kadar yükleniyor?
cevap          : Yemek kartı günlük 250 TL olarak yüklenir. [Parça 4]
skor           : 0.8415716290473938
kaynak         : [4, 5, 2]
model_cagrildi : True


## 5. Adım - Senaryo 1: Belge içi sorular

Cevabı belgede kesinlikle olan sorular soruyorum. Her soru için cevabın içermesi
gereken anahtar ifadeyi de yazdım, böylece **doğruluğu otomatik** kontrol edebiliyorum.

Bu kontrolü koymam önemli: "cevap üretildi" ile "cevap doğru" aynı şey değil.

In [7]:
def bilgi_yok_mu(cevap):
    # Modelin cevap vermeyi reddedip reddetmediğini anlar.
    # İlk sürümde sadece cevabın BAŞINA bakıyordum. Ama model bazen
    # "Hayır, bilgi bulunamadı" gibi bir önek koyuyor ve o sürüm bunu
    # reddetme saymayıp halüsinasyon olarak işaretliyordu. Cevaplar tek
    # cümleyle sınırlı olduğu için ifadeyi her yerde aramak daha doğru.
    return "bilgi bulunamadı" in " ".join(cevap.split()).lower()


belge_ici = [
    ("Yemek kartına günlük ne kadar yükleniyor?",  "250",     "250 TL"),
    ("Uzaktan çalışma haftada kaç gün olabilir?",  "iki",     "en fazla 2 gün"),
    ("Eğitim bütçesi ne kadar?",                   "15.000",  "yıllık 15.000 TL"),
    ("Parola en az kaç karakter olmalı?",          "on iki",  "12 karakter"),
    ("Maaşlar ne zaman ödeniyor?",                 "son iş",  "ayın son iş günü"),
    ("Ekipman yenileme süresi nedir?",             "dört",    "4 yıl"),
    ("Evlilik izni kaç gündür?",                   "3",       "3 gün"),
    ("Mazeret izni yılda toplam kaç gündür?",      "5",       "5 gün"),
]

ici_uretilen, ici_dogru = 0, 0
for soru, anahtar, beklenen in belge_ici:
    r = cevapla(soru)
    uretildi = not bilgi_yok_mu(r["cevap"])
    dogru = uretildi and anahtar.lower() in r["cevap"].lower()
    ici_uretilen += uretildi
    ici_dogru += dogru

    durum = "DOĞRU" if dogru else ("YANLIŞ" if uretildi else "REDDEDİLDİ")
    print(f"[{durum:10}] {soru}")
    print(f"             beklenen: {beklenen}")
    print(f"             cevap   : {' '.join(r['cevap'].split())}")
    print(f"             skor={r['skor']:.4f}  kaynak={r['kaynak']}")
    print()

print(f"SONUÇ: {ici_uretilen}/{len(belge_ici)} cevap üretildi, "
      f"{ici_dogru}/{len(belge_ici)} doğru")

[DOĞRU     ] Yemek kartına günlük ne kadar yükleniyor?
             beklenen: 250 TL
             cevap   : Yemek kartı günlük 250 TL olarak yüklenir. [Parça 4]
             skor=0.8416  kaynak=[4, 5, 2]



[DOĞRU     ] Uzaktan çalışma haftada kaç gün olabilir?
             beklenen: en fazla 2 gün
             cevap   : Uzaktan çalışma haftada en fazla iki gün olarak uygulanır. [Parça 1]
             skor=0.8669  kaynak=[1, 0, 2]



[DOĞRU     ] Eğitim bütçesi ne kadar?
             beklenen: yıllık 15.000 TL
             cevap   : Yıllık eğitim bütçesi her çalışan için 15.000 TL dir. [Parça 4]
             skor=0.8397  kaynak=[5, 4]

[REDDEDİLDİ] Parola en az kaç karakter olmalı?
             beklenen: 12 karakter
             cevap   : Bilgi bulunamadı
             skor=0.7865  kaynak=None



[DOĞRU     ] Maaşlar ne zaman ödeniyor?
             beklenen: ayın son iş günü
             cevap   : Her ayın son iş gününe [Parça 3]
             skor=0.8374  kaynak=[3, 4, 2]



[DOĞRU     ] Ekipman yenileme süresi nedir?
             beklenen: 4 yıl
             cevap   : Dört yıl [Parça 6]
             skor=0.8540  kaynak=[6, 2, 0]



[DOĞRU     ] Evlilik izni kaç gündür?
             beklenen: 3 gün
             cevap   : Evlilik izni 3 gündür. [Parça 2]
             skor=0.8478  kaynak=[2, 3, 1]



[DOĞRU     ] Mazeret izni yılda toplam kaç gündür?
             beklenen: 5 gün
             cevap   : 5 gün [Parça 2]
             skor=0.8729  kaynak=[2, 3]

SONUÇ: 7/8 cevap üretildi, 7/8 doğru


8 sorudan 7'sine cevap üretildi ve **7'si de doğru**. Cevaplar kısa, bağlamdaki
ifadeyi kullanıyor ve kaynak parça numarası veriyor; yani kaynak gösterme kuralı
çalışıyor.

**Kaçan tek soru:** "Parola en az kaç karakter olmalı?" reddedildi. Sebebi model değil,
**eşik**: skoru 0.7865 çıktı, eşik ise 0.8103. Yani 0.024 farkla elendi ve model hiç
çağrılmadı. Cevap belgede duruyor.

Bu bir **yanlış negatif** ve eşik tabanlı filtrelemenin kaçınılmaz bedeli. Eşiği
düşürsem bu soru kurtulur ama belge dışı sorular içeri sızmaya başlar. İlk notebook'ta eşiği
belirlerken kullandığım soru kümesinde bu soru yoktu; yeni bir soru o aralığa denk
gelince yanlış tarafta kaldı. Eşiğin kaç soruyla ölçüldüğü, ne kadar güvenilir olduğunu
doğrudan belirliyor.

## 6. Adım - Senaryo 2: Belge dışı sorular (halüsinasyon testi)

Asıl kritik test bu. Cevabı belgede **olmayan** sorular soruyorum. Sistem bir şey
uydurmamalı, "Bilgi bulunamadı" demeli.

Soruları üç zorlukta seçtim:

- **Tamamen alakasız:** hava durumu, Python
- **Konuyla ilgili ama belgede yok:** şirketin cirosu, çalışan sayısı
- **Tuzak sorular:** belgedeki konunun *yanındaki* bilgiyi soran sorular. En zoru
  bunlar: hem arama yüksek skor verip eşiği geçiyor, hem de model bağlamdaki benzer
  bilgiyi cevap sanabiliyor.

In [8]:
belge_disi = [
    "İstanbul'da yarın hava nasıl olacak?",
    "Python'da bir listeyi nasıl sıralarım?",
    "Şirkette kaç kişi çalışıyor?",
    "Şirketin 2027 ciro hedefi nedir?",
    "Yemek kartı hangi bankadan veriliyor?",
    "Yıllık izin parası ne zaman ödeniyor?",
    "Doğum izni anneler için kaç gündür?",
    "Eğitim bütçesi yurt dışı konferans için kullanılabilir mi?",
]

disi_red, esikte_durdu = 0, 0
for soru in belge_disi:
    r = cevapla(soru)
    reddedildi = bilgi_yok_mu(r["cevap"])
    disi_red += reddedildi
    esikte_durdu += (not r["model_cagrildi"])

    isaret = "OK     " if reddedildi else "UYDURDU"
    nerede = "eşikte" if not r["model_cagrildi"] else "istemde"
    print(f"[{isaret}] {soru}")
    print(f"           skor={r['skor']:.4f}  durduran={nerede if reddedildi else '-'}")
    print(f"           cevap: {' '.join(r['cevap'].split())}")
    print()

print(f"SONUÇ: {disi_red}/{len(belge_disi)} soruda 'Bilgi bulunamadı'")
print(f"       {esikte_durdu}/{len(belge_disi)} tanesi modele hiç gitmeden eşikte durduruldu")

[OK     ] İstanbul'da yarın hava nasıl olacak?
           skor=0.7493  durduran=eşikte
           cevap: Bilgi bulunamadı

[OK     ] Python'da bir listeyi nasıl sıralarım?
           skor=0.7749  durduran=eşikte
           cevap: Bilgi bulunamadı

[OK     ] Şirkette kaç kişi çalışıyor?
           skor=0.8088  durduran=eşikte
           cevap: Bilgi bulunamadı

[OK     ] Şirketin 2027 ciro hedefi nedir?
           skor=0.7782  durduran=eşikte
           cevap: Bilgi bulunamadı



[OK     ] Yemek kartı hangi bankadan veriliyor?
           skor=0.8157  durduran=istemde
           cevap: Bilgi bulunamadı [Parça 4].



[UYDURDU] Yıllık izin parası ne zaman ödeniyor?
           skor=0.8647  durduran=-
           cevap: Maaş ödemeleri her ayın son iş gününe tabidir. [Parça 3]



[OK     ] Doğum izni anneler için kaç gündür?
           skor=0.8474  durduran=istemde
           cevap: Bilgi bulunamadı [Parça 2].



[OK     ] Eğitim bütçesi yurt dışı konferans için kullanılabilir mi?
           skor=0.8439  durduran=istemde
           cevap: Hayır, bilgi bulunamadı [Parça 5].

SONUÇ: 7/8 soruda 'Bilgi bulunamadı'
       4/8 tanesi modele hiç gitmeden eşikte durduruldu


8 sorudan 7'si doğru şekilde reddedildi. Ama asıl öğretici olan **nerede**
durduruldukları:

| Nerede durdu | Kaç soru | Hangileri |
|---|---|---|
| Skor eşiğinde (model hiç çalışmadı) | 4 | hava durumu, Python, çalışan sayısı, ciro hedefi |
| İstemdeki kuralda | 3 | yemek kartı bankası, anne doğum izni, yurt dışı konferans |
| Hiçbiri (uydurdu) | 1 | yıllık izin parası |

Bu tablo **iki savunma hattının da gerekli olduğunu** gösteriyor. Sadece eşiğe
güvenseydim 3 soru daha cevaplanacaktı; sadece istemdeki kurala güvenseydim 4 soru için
boşuna model çalıştırılacaktı.

Dikkat çeken bir nokta: eşikte durdurulan 4 soru belgeyle hiç ilgisi olmayanlar, istemde
durdurulanlar ise belgedeki bir konunun **yanındaki** bilgiyi soranlar. Yani skor eşiği
alakasızı eliyor, ama anlamsal olarak yakın olanı elemiyor — çünkü onlar gerçekten
yakın. Gün 14'te "vektör araması her zaman bir sonuç döndürür" diye not ettiğim durumun
pratik hali bu.

**Kaçan soru:** "Yıllık izin parası ne zaman ödeniyor?" sorusuna model "Maaş ödemeleri
her ayın son iş gününe tabidir" dedi. Belgede *maaş* ödemesinin ne zaman yapıldığı
yazıyor, ama sorulan şey kullanılmayan **izin bedelinin** ödenmesi. İkisi farklı şeyler.
Model en yakın bilgiyi alıp cevap olarak sundu.

Halüsinasyonun ince hali bu: uydurma bir cümle değil, doğru bir bilginin sorulmayan bir
soruya cevap gibi sunulması. Bağlamda geçen bir ifadeyi kullandığı için "kaynağa sadık"
görünüyor.

Bir de kendi test yöntemimde bir kusur buldum. Reddetmeyi ilk yazdığımda cevabın
**başında** "Bilgi bulunamadı" arıyordum. Model son soruda `"Hayır, bilgi bulunamadı"`
demiş; bu aslında bir reddetme ama ilk sürüm onu halüsinasyon saymıştı. Kontrolü ifadeyi
her yerde arayacak şekilde düzeltince skor 6/8'den 7/8'e çıktı. Yani bir noktada
ölçtüğüm şey sistemin başarısı değil, testimin kalitesiydi.

## 7. Adım - Sonuç tablosu

In [9]:
print(f"{'senaryo':44} {'test':>5} {'sonuç':>9}")
print("-" * 62)
print(f"{'Belge içi: cevap üretildi':44} {len(belge_ici):>5} {f'{ici_uretilen}/{len(belge_ici)}':>9}")
print(f"{'Belge içi: DOĞRU bilgi içeriyor':44} {len(belge_ici):>5} {f'{ici_dogru}/{len(belge_ici)}':>9}")
print(f"{'Belge dışı: Bilgi bulunamadı':44} {len(belge_disi):>5} {f'{disi_red}/{len(belge_disi)}':>9}")
print(f"{'  -> eşikte durduruldu (model çalışmadı)':44} {len(belge_disi):>5} {f'{esikte_durdu}/{len(belge_disi)}':>9}")
print()
print(f"embedding modeli : {EMBED_MODEL}")
print(f"üretim modeli    : {URETIM_MODEL} (Ollama, 4-bit, temperature=0)")
print(f"skor eşiği       : {ESIK:.4f}")
print(f"çalışma yeri     : tamamen yerel, internet gerekmiyor")

senaryo                                       test     sonuç
--------------------------------------------------------------
Belge içi: cevap üretildi                        8       7/8
Belge içi: DOĞRU bilgi içeriyor                  8       7/8
Belge dışı: Bilgi bulunamadı                     8       7/8
  -> eşikte durduruldu (model çalışmadı)         8       4/8

embedding modeli : intfloat/multilingual-e5-base
üretim modeli    : qwen2.5:7b-instruct (Ollama, 4-bit, temperature=0)
skor eşiği       : 0.8103
çalışma yeri     : tamamen yerel, internet gerekmiyor


---

## Değerlendirme

### Kurduğum sistem

Hazır hiçbir RAG kütüphanesi kullanmadan, uçtan uca ve tamamen yerel çalışan bir RAG
hattı yazdım:

| Adım | Nasıl yazdım |
|---|---|
| Parçalama | Özyinelemeli bölme + örtüşme, kendi fonksiyonum (gün 14) |
| Vektörleştirme | `multilingual-e5-base`, 768 boyut, yerel |
| Depolama | numpy matrisi, normalize edilmiş, `.npz` olarak diskte |
| Arama | Saf numpy kosinüs benzerliği, tek matris çarpımı (gün 13) |
| Filtreleme | Ölçüme dayalı skor eşiği |
| İstem | Bağlam etiketleri + sadakat kuralı + kaçış yolu + kaynak gösterme |
| Üretim | `qwen2.5:7b-instruct` (Ollama, 4-bit), `temperature=0` |

### Sonuçlar

| Senaryo | Sonuç |
|---|---|
| Belge içi (8 soru) | 7 cevap üretildi, **7'si doğru** |
| Belge dışı (8 soru) | **7 reddedildi**, 1 halüsinasyon |
| Eşikte durdurulan | 4 (bu sorular için model hiç çalışmadı) |
| İstem kuralında durdurulan | 3 |

### Öğrendiğim en önemli şeyler

**Hata katman katman ayrışıyor.** İki hatam iki farklı yerden geldi: biri eşikten (doğru
soru elendi), biri modelden (yakın bilgiyi cevap sandı). Hangi katmanın hata verdiğini
ayırt etmeden "sistem çalışmıyor" demek anlamsız, çünkü çözümleri de farklı: biri eşik
ayarı, diğeri istem veya model meselesi.

**Skor eşiği alakasızı eler, yakını elemez.** Reddedilen 8 sorudan 4'ü eşikte durdu ama
belgedeki konunun yanındaki bilgiyi soran 3 soru eşiği rahat geçti; onları istemdeki
kural durdurdu. İki hattın da gerekli olmasının sebebi bu.

**Kesinlik ve kapsam arasında seçim yapmak zorundayım.** Eşiği yükseltirsem halüsinasyon
azalır ama doğru sorular elenir; düşürürsem tersi olur. Aynı ödünleşimi gün 14'te
parçalama örtüşmesinde de görmüştüm: az örtüşme bilgi kaybı, çok örtüşme maliyet. RAG'de
tek bir "doğru ayar" yok; hangi hatanın daha az zararlı olduğuna karar veriliyor.

**Sıcaklık sıfır tekrarlanabilirlik garantisi değil.** Aynı soruyu iki oturumda sorunca
farklı cevaplar aldım; sebebi örnekleme değil, kayan nokta işlemlerinin sırası.

**Otomatik test kolayca yanıltıyor.** Reddetme kontrolümün ilk sürümü doğru bir cevabı
halüsinasyon saymıştı. Değerlendirme kodunun kendisi de test edilmesi gereken bir şey.

### Eksik kalanlar

- **Yeniden sıralama (re-ranking) yok.** Gün 14'te en yüksek getirili iyileştirmenin bu
  olduğunu öğrenmiştim, bu hatta eklemedim.
- **Tek belge, tek kaynak.** Gerçek sistemde yönlendirme (routing) gerekirdi.
- **Eşik 8 soruyla ölçüldü.** Kaçan sorunun sebebi de buydu; daha geniş bir kümeyle
  ölçülmeliydi.
- **Değerlendirme kümesi küçük** (16 soru). Gün 13'te "AI Mühendisi değerlendirme
  altyapısı kurar" diye yazmıştım; burada yaptığım onun en ilkel hali.

### Yerel model tercihi

Bu hattı önce bir bulut API'siyle kurmuştum. Ücretsiz katmanın günlük istek sınırı
testleri tekrarlamayı imkansız kılınca yerele geçtim. Görevlendirme "bir API kullanarak"
dediği için bu tercih görevin lafzından sapıyor; RAG mantığının kendisi ise değişmedi:
parçalama, benzerlik hesabı, arama ve istem tamamen kendi yazdığım kod, yasak olan hazır
RAG kütüphanelerinden hiçbiri kullanılmadı.

### Staj boyunca öğrendiklerimin bu projeye çıkması

- **Gün 13** – kosinüs benzerliği, normalize edip iç çarpımla hızlı arama
- **Gün 14 (RAG mimarileri)** – Naive RAG hattının adımları ve tıkanıklıkları
- **Gün 14 (temel bileşenler)** – özyinelemeli parçalama, örtüşme, skor eşiği, halüsinasyon önlemleri
- **Gün 14 (örnekleme)** – sıcaklık kararı ve gerekçesi

Gün 13'te "AI Mühendisi hazır modeli güvenilir bir ürüne dönüştürür" diye yazmıştım.
Bu proje o işin küçük bir örneği oldu: model hazır, asıl iş etrafına kurulan sistemde.

## Sonuç

Görevlendirmenin istediği üç çıktıyı da ürettim:

1. **Araştırma raporu** — gün 13-14'teki altı bölüm
2. **Native RAG kodu** — bu notebook ve ilk notebook, hiçbir RAG framework'ü
   içermiyor
3. **Test çıktıları** — belge içi sorularda doğru bilginin getirildiğini, belge dışı
   sorularda "Bilgi bulunamadı" cevabı verildiğini gösteren senaryolar

RAG halüsinasyonu azaltıyor ama tek başına bitirmiyor; skor eşiği ile istemdeki kuralın
ikisi birden gerekiyor.